In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: clean and split, then for each fold fit the encoder and phrases once,
# train `EMBEDDING_MODEL`, and save `{EMBEDDING_MODEL}_para_{fold}`.
# False: skip that work. The tuning loop loads those saved fold frames.
REBUILD = False
TRAIN = True

spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
ANALYTICS_MODEL = "lr_improved"
EMBEDDING_MODEL = "glove"
K = 5

PARA_RANGE = dict()

In [ ]:
from pyspark.sql import functions as F

from src.cleaning import data_cleaning
from src.embeddings import EmbeddingModel
from src.encoder import encoder
from src.split import split_and_weight
from src.text_features import text_feature_engineering, fit_phrases


if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned, True)
    df_engineered.cache().count()
    data_oi.unpersist(df_cleaned)
    del df_raw, df_cleaned

    df_train, df_valid, df_test = split_and_weight(df_engineered, SEED)
    data_oi.unpersist(df_engineered, df_valid, df_test)
    del df_engineered, df_valid, df_test
    # The training frame already occupies hash buckets 0–6999 of `SEED`.
    # Fold on another seed so the K parts stay inside that frame.
    fold_parts = list(split_and_weight(df_train.drop("weight"), SEED + 1, (1,) * K))
    data_oi.unpersist(df_train)
    del df_train
    unweighted_first = fold_parts[0].drop("weight").cache()
    unweighted_first.count()
    data_oi.unpersist(fold_parts[0])
    fold_parts[0] = unweighted_first
    del unweighted_first

    for fold in range(K):
        print(f"Fold {fold}")
        holdout_rows = fold_parts[fold]
        others = [part for index, part in enumerate(fold_parts) if index != fold]
        fit_base = others[0]
        for part in others[1:]:
            fit_base = fit_base.unionByName(part)
        fit_base = fit_base.cache()
        n = fit_base.count()
        class_n = fit_base.select("rating").distinct().count()
        weights = (
            fit_base.groupBy("rating").count()
            .withColumn("weight", F.lit(n) / (F.lit(class_n) * F.col("count")))
            .select("rating", "weight")
        )
        fit_rows = fit_base.join(weights, "rating", "left").cache()
        fit_rows.count()

        encoded_fit, encoded_holdout, encoded_empty = encoder(
            fit_rows, holdout_rows, fit_rows.limit(0)
        )
        encoded_fit.count()
        encoded_holdout.count()
        encoded_empty.count()

        phrases = fit_phrases(encoded_fit)
        embedding_model = EmbeddingModel(
            encoded_fit, encoded_holdout, encoded_empty, phrases
        )
        model_name = f"{EMBEDDING_MODEL}_para_{fold}"
        print(f"Fold {fold} Embedding Model `{EMBEDDING_MODEL}` Training")
        EmbeddingModel.train_registry[EMBEDDING_MODEL](embedding_model, model_name)
        print(f"Fold {fold} Embedding Model `{EMBEDDING_MODEL}` Running")
        train_embedding, valid_embedding, test_embedding = EmbeddingModel.predict_registry[EMBEDDING_MODEL](
            embedding_model, model_name
        )
        data_oi.save_embedding_data(
            model_name, train_embedding, valid_embedding, test_embedding
        )
        fold_cache = [
            fit_base, fit_rows,
            encoded_fit, encoded_holdout, encoded_empty,
            train_embedding, valid_embedding, test_embedding,
        ]
        for extra in (
            embedding_model._train_expanded,
            embedding_model._train_expanded_phrase,
        ):
            if extra is not None:
                fold_cache.append(extra)
        data_oi.unpersist(*fold_cache)

    data_oi.unpersist(*fold_parts)
    del fold_parts

In [ ]:
from itertools import product

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

from src.analytics import AnalyticsModel

analytics_function = AnalyticsModel.registry[ANALYTICS_MODEL]
range_keys = list(PARA_RANGE)
candidates = [
    dict(zip(range_keys, values))
    for values in product(*(PARA_RANGE[key] for key in range_keys))
]
evaluator = RegressionEvaluator(labelCol="rating", predictionCol="rating_pred", metricName="mae")

In [ ]:
best_para = None
best_mae = None

for trial in candidates:
    fold_maes = []

    for fold in range(K):
        model_name = f"{EMBEDDING_MODEL}_para_{fold}"
        fit_embedding, holdout_embedding, empty_embedding = data_oi.load_embedding_data(model_name)
        model = AnalyticsModel(
            EMBEDDING_MODEL, fit_embedding, holdout_embedding, empty_embedding
        )
        _, holdout_pred, _ = analytics_function(model, trial)
        scored = holdout_pred.withColumn("rating_pred", F.col("prediction") + 1)
        fold_maes.append(evaluator.evaluate(scored))
        model.release_features()
        data_oi.unpersist(
            fit_embedding, holdout_embedding, empty_embedding, holdout_pred
        )
        del fit_embedding, holdout_embedding, empty_embedding, holdout_pred, scored, model
    
    mae = sum(fold_maes) / K
    print(trial, mae)
    if best_mae is None or mae < best_mae:
        best_mae = mae
        best_para = trial

data_oi.save_analytics_para(EMBEDDING_MODEL, ANALYTICS_MODEL, best_para)
para = data_oi.load_analytics_para(EMBEDDING_MODEL, ANALYTICS_MODEL)
print(f"best mae: {best_mae}")
print(para)